In [0]:
import pyspark.sql.functions as F

# 1. Define parameters for catalog and rows (leave batch_number out of widgets)
dbutils.widgets.text("catalog", "meridian_tNN")
dbutils.widgets.text("rows_per_batch", "2000")

catalog = dbutils.widgets.get("catalog")
rows = int(dbutils.widgets.get("rows_per_batch"))

volume_path = f"/Volumes/{catalog}/landing/raw_files"
dimensions = ["reason", "store", "item", "customer"]

# 2. Auto-detect the next batch number based on existing files in the volume
def get_next_batch():
    try:
        # Check existing batches in the store_returns folder
        existing = dbutils.fs.ls(f"{volume_path}/store_returns")
        batch_nums = []
        for f in existing:
            if "batch_" in f.name:
                num = f.name.replace("batch_", "").replace("/", "")
                if num.isdigit():
                    batch_nums.append(int(num))
        return max(batch_nums) + 1 if batch_nums else 1
    except Exception:
        # If the folder doesn't exist at all, this is batch 1
        return 1

batch = get_next_batch()
print(f"Automatically detected! Running simulator for Batch: {batch}")

# Land dimensions and lookup table once during batch 1
if batch == 1:
    for dim in dimensions:
        df_dim = spark.table(f"samples.tpcds_sf1.{dim}")
        df_dim.write.mode("overwrite").parquet(f"{volume_path}/{dim}/batch_1")
    
    # Store sales is a read-only lookup
    df_sales = spark.sql("SELECT * FROM samples.tpcds_sf1.store_sales LIMIT 10000")
    df_sales.write.mode("overwrite").parquet(f"{volume_path}/store_sales/batch_1")

# Inject SCD2 updates for the 'reason' dimension in later batches
elif batch == 2:
    # Update some reason descriptions to trigger CDC
    df_reason = spark.table("samples.tpcds_sf1.reason") \
        .withColumn("r_reason_desc", 
                    F.when(F.col("r_reason_sk") == 1, F.lit("UPDATED BATCH 2: Defective product"))
                     .otherwise(F.col("r_reason_desc")))
    df_reason.write.mode("overwrite").parquet(f"{volume_path}/reason/batch_2")

elif batch == 3:
    # A second round of updates to trigger more CDC
    df_reason = spark.table("samples.tpcds_sf1.reason") \
        .withColumn("r_reason_desc", 
                    F.when(F.col("r_reason_sk") == 1, F.lit("UPDATED BATCH 3: Severely defective product"))
                     .when(F.col("r_reason_sk") == 2, F.lit("UPDATED BATCH 3: Wrong size sent"))
                     .otherwise(F.col("r_reason_desc")))
    df_reason.write.mode("overwrite").parquet(f"{volume_path}/reason/batch_3")

# Slice fact data (store_returns) into disjoint batches
offset = (batch - 1) * rows
df_fact = spark.sql(f"""
    SELECT * FROM samples.tpcds_sf1.store_returns 
    ORDER BY sr_returned_date_sk 
    LIMIT {rows} OFFSET {offset}
""")

# Inject bad data (null keys, duplicates, invalid values) for Silver to catch later
df_bad_data = df_fact.limit(3) \
    .withColumn("sr_item_sk", F.lit(None).cast("int")) \
    .withColumn("sr_return_quantity", F.lit(-5))

df_final_batch = df_fact.union(df_bad_data)

# Write the fact batch
df_final_batch.write.mode("overwrite").parquet(f"{volume_path}/store_returns/batch_{batch}")